In [2]:
# Edited by Sarah Hayduk (SH)
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# (SH) import dash callback_context "advanced callback" to handle the buttons Water/Mountain/Disaster/Reset
from dash import callback_context

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# (SH) import CRUD Python module file name and class name
from PythonModule_Hayduk import CRUD

###########################
# Data Manipulation / Model
###########################
# (SH) remove parameters. all parameters hardcoded into Python module
#username = "aacuser"
#password = "Hayduk"

# (SH) Connect to database via CRUD Module
db = CRUD()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# (SH) Add in Grazioso Salvare’s logo (in project folder)
image_filename = 'Grazioso Salvare Logo.png'
#read file and encode as base64 so it can be embedded in the HTML
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# (SH) Dashboard title with unique identifier and clickable logos with tooltips
app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    # (SH) logos and title section (side by side horizontally)
    html.Div([
        # logo to the left (grazioso image) 
        #clickable to snhu page. link opens in new tab with tooltip on hover
        html.A(
            href = 'https://www.snhu.edu',
            target = 'blank',
            title = 'SNHU Home Page',
            children = html.Img(
                src = 'data:image/png;base64,{}'.format(encoded_image.decode()),
                id = 'logo-left',
                #scale image, prevent from stretching within flex container
                style = {
                    'height': '150px',
                    'margin-right': '20px',
                    'flex': '0 0 auto'
            })
        ),
        # title in the middle (unique identifier)
        html.Div([
            html.H1("Sarah's MongoDB Dashboard",
                    style={
                        'color': 'rgb(145, 0, 115)', 
                        'margin': '0'}),
            #subtitle with course/project info
            html.P("CS340 Project Two: Sarah Hayduk",
                  style={
                      'fontSize': '16px',
                      'color': 'black',
                      'marginTop': '4px',
                      'fontStyle': 'italic'
                  })
        #stack the title and subtitle vertically in the center
        ], style={
            'display': 'flex',
            'flexDirection': 'column',
            'alignItems': 'center'
            } 
        ),
        # logo to the right (grazioso image)
        # also clickable to snhu page. link opens in new tab with tooltip on hover
        html.A(
            href = 'https://snhu.edu',
            target = 'blank',
            title = 'SNHU Home Page',
            children = html.Img(
                src = 'data:image/png;base64,{}'.format(encoded_image.decode()),
                id = 'logo-right',
                #scale image, prevent from stretching within flex container
                style={
                    'height': '150px',
                    'margin-right': '20px',
                    'flex': '0 0 auto'
            })
        )
    #flex container styling for logos and title section
    #center itmes horizontally, align item vertically, add spacing between items
    ], style={
        'display': 'flex',
        'justifyContent': 'center',
        'alignItems': 'center',
        'gap': '12px',
        'margin-top': '10px',
        'paddingLeft': '20px',
        'paddingRight': '20px'
    }),
    html.Hr(),
    
    # (SH) add button filter options and status header for rescue type
    html.Div(
        # arrange buttons and header horizontally starting at the left
        style={
            'display': 'flex',
            'alignItems': 'center',
            'justifyContent': 'flex-start',
            'gap': '20px',
            'marginBottom': '10px'
        },
        #buttons and header elements
        children=[
            #row of buttons for user to filter rescue type
            html.Div(className='buttonRow',
            style={
                'display' : 'flex', 
                'gap': '10px'
            },
            children=[
                html.Button(id='submit-button-one', n_clicks=0, children='Water Rescue'),
                html.Button(id='submit-button-two', n_clicks=0, children='Mountain Rescue'),
                html.Button(id='submit-button-three', n_clicks=0, children='Disaster Rescue'),
                html.Button(id='submit-button-four', n_clicks=0, children='Reset')
                ]
            ),
            #status header that updates with each click to describe what is being displayed
            html.Div(id='rescue-type-header', style={
                'fontWeight': 'bold',
                'color': 'black',
                'fontSize': '14px'
            })
        ]),
        
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         
                         # (SH) features for interactive data table to make it user-friendly for the client
                         #copied from module 6 assignment
                         editable=False,
                         filter_action="native",
                         sort_action="native",
                         column_selectable=False,
                         row_selectable="single", #enable single row selection for mapping callback
                         row_deletable=False,
                         selected_columns=[],
                         selected_rows=[0], #default to first row on load
                         page_action="native",
                         page_current=0,
                         page_size=10, #10 results per page
                         
                         # (SH) customizations to header and cells for personality and readability
                         style_header={
                             'backgroundColor': 'rgb(145, 0, 115)', #purple
                             'color': 'white', 
                             'fontWeight': 'bold',
                             'fontSize': '14px' #slightly larger than data cell font
                         },
                         style_data={
                             'color': 'black',
                             'backgroundColor': 'white'
                         },
                         style_cell={
                             'textAlign': 'left',
                             'padding': '5px'
                         }),
    html.Br(),
    html.Hr(),
    #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    # (SH) justify chart and map so they stay centered when page zooms out
    html.Div(className='row',
         style={'display' : 'flex',
               'justifyContent': 'center',
               'alignItems': 'center'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ]),
    
    # (SH) add a footer with unique identifier and logo, why not also clickable to snhu
    html.Div(
        #horizontal layout for text and logo side by side
        style={
            'display': 'flex',
            'justifyContent': 'center',
            'alignItems': 'center',
            'marginTop': '20px',
            'paddingLeft': '40px',
            'paddingRight': '40px',
            'gap': '20px'
        },
        #text with my name and course details
        children = [
            html.Div(
                children="© 2025 Sarah Hayduk — CS340 Client/Server Development",
                style = {
                    'color': '#910073', 
                    'fontSize': '24px',
                    'fontWeight': 'bold',
                }
            ),
            #another smaller clickable logo to snhu homepage
            html.A(
                href = 'https://snhu.edu',
                target = 'blank',
                title = 'SNHU Home Page',
                children = html.Img(
                    src = 'data:image/png;base64,{}'.format(encoded_image.decode()),
                    id = 'foot_logo',
                    style={'height': '100px'}
                )
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# (SH) This callback will make the buttons for Water/Mountain/Disaster/Reset function for easy sorting
@app.callback(
    Output('datatable-id', 'data'),
    Output('rescue-type-header', 'children'),  #status header
    Output('datatable-id', 'filter_query'),    #column typed filters clear
    Output('datatable-id', 'sort_by'),         #resets the sort arrows
    Output('datatable-id', 'selected_rows'),   #resets selected row to first
    [Input('submit-button-one', 'n_clicks'),   #water
     Input('submit-button-two', 'n_clicks'),   #mountain
     Input('submit-button-three', 'n_clicks'), #disaster
     Input('submit-button-four', 'n_clicks')]  #reset
)
# (SH) Using the dash advanced context_callback to determine which input triggered the callback
# When rescue filter button is clicked, update status header and retrieve dogs matching each rescue criteria.
#on each button press, clear typed input in columns, clear column sorting arrows, and return selected row to first.
#on Reset, restore dashboard to its original unfilted state.
def on_click(water, mountain, disaster, reset):
    #get ID of the button that was most recently clicked
    triggered_id = callback_context.triggered[0]['prop_id'].split('.')[0]
    
    #if water, retrieve all dogs matching query for dogs suited to water rescue
    if triggered_id == 'submit-button-one':
        header_text = "Displaying Water Rescue Dogs" #update status header
        df = pd.DataFrame.from_records(db.read({
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
            '$or': [
                #use regex for specific dog breeds using ^...$ to anchor strings. $options:i for case-insensitivity
                {'breed': {'$regex': '^Labrador Retriever Mix$', '$options': 'i'}},
                {'breed': {'$regex': '^Chesa Bay Retr$', '$options': 'i'}}, #Cheasapeake Bay Retriever in data set
                {'breed': {'$regex': '^Newfoundland$', '$options': 'i'}},
                #use regex, no anchor, for Lab mix inclusivity. $options:i for case-insensitivity
                {'breed': {'$regex': 'Labrador Retriever/', '$options': 'i'}},
                {'breed': {'$regex': '/Labrador Retriever', '$options': 'i'}}
            ]
        }))
    #else if mountain, retrieve all dogs matching query suited to mountain(wilderness) rescue
    elif triggered_id == 'submit-button-two':
        header_text = "Displaying Mountain Rescue Dogs" #update status header
        df = pd.DataFrame.from_records(db.read({
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
            '$or': [
                #use regex for specific dog breeds using ^...$ to anchor strings. $options:i for case-insensitivity
                {'breed': {'$regex': '^German Shepherd$', '$options': 'i'}},
                {'breed': {'$regex': '^Alaskan Malamute$', '$options' :'i'}},
                {'breed': {'$regex': '^Old English Sheepdog$', '$options': 'i'}},
                {'breed': {'$regex': '^Siberian Husky$', '$options': 'i'}},
                {'breed': {'$regex': '^Rottweiler$', '$options': 'i'}}
            ]
        }))
    #else if disaster, retrieve all dogs matching query suited for disaster(tracking) rescue
    elif triggered_id == 'submit-button-three':
        header_text = "Displaying Disaster Rescue Dogs" #update status header
        df = pd.DataFrame.from_records(db.read({
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300},
            '$or': [
                ##use regex for specific dog breeds using ^...$ to anchor strings. $options:i for case-insensitivity
                {'breed': {'$regex': '^Doberman Pinsch$', '$options': 'i'}}, #Doberman Pinscher in data set
                {'breed': {'$regex': '^German Shepherd$', '$options': 'i'}},
                {'breed': {'$regex': '^Golden Retriever$', '$options': 'i'}},
                {'breed': {'$regex': '^Bloodhound$', '$options': 'i'}},
                {'breed': {'$regex': '^Rottweiler$', '$options': 'i'}}
            ]
        }))
    #else, reset retrieves all animals (unfiltered)
    else:
        header_text = "Displaying All Animals" #update status header
        df = pd.DataFrame.from_records(db.read({}))
    
    #cleanup
    df.drop(columns=['_id'], inplace=True)
    #return the filtered data, update the header text, clear column filters,
    #clear sorting arrows, set selected row to first
    return df.to_dict('records'), header_text, '', [], [0]


# (SH) Interactive Pie Chart 
#Display the breeds of animal based on quantity represented in the current data table
#tooltip on hover displays breed name and quantity
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])

def update_graphs(viewData):
    # prevent errors on initial load or empty table
    if viewData is None or len(viewData) == 0:
        return html.Div("No map data available.")
    
    df = pd.DataFrame(viewData)
    
    # count occurance of each breed
    breed_counts = df['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    
    fig = px.pie(
        breed_counts.head(8), #limit 8
        names = 'breed', #slices by breed
        values = 'count', #slice proportion by count
        # manually change the colors of the pie slices, i did not like the default
        color_discrete_sequence=[
            '#003F5C', #deep blue
            '#58508D', #muted purple
            '#8A508F', #plum
            '#Bc5090', #magenta
            '#DE5A79', #rose
            '#FF6361', #coral
            '#FF8531', #orange
            '#FFA600', #gold
        ]
    )
    #display percentage on chart
    fig.update_traces(textinfo='percent')
    #return graph, adjust size so it looks better next to map
    return [dcc.Graph(figure=fig, style={
        'height': '500px',
        'width': '100%',
        'margin': 'auto'
    })]
    
    
# (SH) This callback will highlight a row on the data table when the user selects it.
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')] #indices relative to fultered data
)

# highlight the selected row in peach
def update_styles(viewData, selected_rows):
    if selected_rows is None or viewData is None:
        return[]
    
    return [{
        'if': {'row_index': i},
        'backgroundColor': '#FFEBDC'
    }    for i in selected_rows]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])

def update_map(viewData, index):  
    # prevent errors on initial load or empty table
    if viewData is None or len(viewData) == 0:
        return html.Div("No map data available.")
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]

app.run_server(debug=True)

Dash app running on http://127.0.0.1:17769/
